# Real-Time Hand Gesture Classification with MobileNet and OpenCV

This project builds an end-to-end computer vision pipeline for classifying rock-paper-scissors hand gestures from images and webcam input. It compares a custom convolutional neural network baseline against a MobileNet transfer learning model, then extends the best-performing approach toward real-time webcam inference using OpenCV.

The project was originally developed from a university computer vision assignment and later refactored into a standalone portfolio project focused on dataset preparation, model comparison, evaluation, and practical deployment.

## Project Goals

The main goals of this project are to:

- Prepare a balanced image dataset for rock-paper-scissors gesture classification.
- Build a custom CNN baseline using PyTorch.
- Apply transfer learning using MobileNet pretrained on ImageNet.
- Compare model performance using accuracy, confusion matrices, and error analysis.
- Explore real-time webcam inference using OpenCV.
- Present the project as a clean, reproducible AI/ML portfolio project.

## Project Workflow

The notebook follows this workflow:

1. Environment setup and configuration
2. Dataset setup and image discovery
3. Stratified train/dev/test split
4. Exploratory data analysis
5. PyTorch dataset and dataloader creation
6. Custom CNN baseline training
7. MobileNet transfer learning
8. Training curve visualization
9. Model evaluation and comparison
10. Error analysis and failure case inspection
11. Real-time webcam inference planning
12. Conclusion and future improvements

## Project Scope

This version focuses on static image classification and real-time frame-by-frame webcam inference. The model predicts one of three gesture classes: rock, paper, or scissors.

This project does not yet perform dynamic gesture recognition, hand tracking, or temporal video classification. These are listed as future improvements.

# 1. Environment Setup and Configuration

## 1.1 Imports and Reproducibility

This section imports the main libraries used throughout the project and sets a fixed random seed. Reproducibility is important in machine learning because dataset splits, model initialization, and training behavior can change between runs if randomness is not controlled.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

## 1.2 Random Seed

A fixed seed is used to make random operations more consistent across runs. This helps keep dataset splitting and model experiments easier to compare.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to: {SEED}")

## 1.3 Device Configuration

The notebook automatically selects the best available device:

- CUDA, if running on an NVIDIA GPU
- MPS, if running on Apple Silicon
- CPU, if no accelerator is available

This makes the notebook portable across different machines.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 1.4 Project Paths

The project uses a consistent folder structure so that data, outputs, and models are easy to manage. The notebook is stored inside the `notebooks/` folder, so paths are defined relative to the project root.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "archive"
SPLITS_DIR = DATA_DIR / "splits"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for directory in [DATA_DIR, SPLITS_DIR, OUTPUTS_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Splits directory: {SPLITS_DIR}")
print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Models directory: {MODELS_DIR}")

## 1.5 Class Labels

The classification task contains three hand gesture classes: rock, paper, and scissors. These labels are mapped to numeric IDs for model training.

In [ ]:
CLASS_NAMES = ["rock", "paper", "scissors"]
CLASS_TO_IDX = {class_name: idx for idx, class_name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: class_name for class_name, idx in CLASS_TO_IDX.items()}

print(CLASS_TO_IDX)